In [2]:
def check_bout_mesh_y_decomposition(p):
    """
    Diagnostic version of checkBoutMeshYDecomposition.

    Prints every region length and modulo check so you can see
    exactly why a given configuration passes or fails.
    """

    num_y_processors = p["num_y_processors"]
    ny = p["ny"]
    num_y_guards = p["num_y_guards"]
    jyseps1_1 = p["jyseps1_1"]
    jyseps2_1 = p["jyseps2_1"]
    jyseps1_2 = p["jyseps1_2"]
    jyseps2_2 = p["jyseps2_2"]
    ny_inner = p["ny_inner"]
    topology = p["topology"]

    print("\n=== INPUT ===")
    for k, v in p.items():
        print(f"{k:15s} = {v}")

    num_local_y_points = ny // num_y_processors
    print(f"\nMYSUB = ny / NYPE = {num_local_y_points}")

    # --- Guard check ---
    if num_local_y_points < num_y_guards and num_y_processors != 1:
        print("FAIL: ny/NYPE < MYG")
        return False

    # --- Lower leg check ---
    #val = jyseps1_1 + 1
    #print(f"\nLower leg: (jyseps1_1 + 1) = {val}")
    #print(f"  {val} % {num_local_y_points} = {val % num_local_y_points}")
    #if val % num_local_y_points != 0:
    #    print("FAIL: lower leg")
    #    return False

    # --- Topology-specific ---
    if jyseps2_1 != jyseps1_2:

        if topology in ("UDN", "CDN"):
            core1 = jyseps2_1 - jyseps1_1
            print(f"\nCore DN1: {core1} % {num_local_y_points} = {core1 % num_local_y_points}")
            if core1 % num_local_y_points != 0:
                print("FAIL: DN core 1")
                return False

            core2 = jyseps2_2 - jyseps1_2
            print(f"Core DN2: {core2} % {num_local_y_points} = {core2 % num_local_y_points}")
            if core2 % num_local_y_points != 0:
                print("FAIL: DN core 2")
                return False

            leg1 = ny_inner - jyseps2_1 - 1
            print(f"Upper leg 1: {leg1} % {num_local_y_points} = {leg1 % num_local_y_points}")
            if leg1 % num_local_y_points != 0:
                print("FAIL: upper leg 1")
                return False

            leg2 = jyseps1_2 - ny_inner + 1
            print(f"Upper leg 2: {leg2} % {num_local_y_points} = {leg2 % num_local_y_points}")
            if leg2 % num_local_y_points != 0:
                print("FAIL: upper leg 2")
                return False

        elif topology == "SF":
            print("\n--- Snowflake checks ---")

            core = jyseps2_1 - jyseps1_1
            print(f"Core: jyseps2_1 - jyseps1_1: {core} % {num_local_y_points} = {core % num_local_y_points}")
            if core % num_local_y_points != 0:
                print("FAIL: SF core")
                return False

            e_leg = jyseps2_2 - ny_inner + 1
            print(f"E leg: jyseps2_2 - ny_inner + 1: {e_leg} % {num_local_y_points} = {e_leg % num_local_y_points}")
            if e_leg % num_local_y_points != 0:
                print("FAIL: SF east leg")
                return False

            n_leg = ny_inner - jyseps1_2 - 1
            print(f"N leg: ny_inner - jyseps1_2 - 1: {n_leg} % {num_local_y_points} = {n_leg % num_local_y_points}")
            if n_leg % num_local_y_points != 0:
                print("FAIL: SF north leg")
                return False

            w_leg = jyseps1_2 - jyseps2_1 - 1
            print(f"W leg: jyseps1_2 - jyseps2_1 - 1: {w_leg} % {num_local_y_points} = {w_leg % num_local_y_points}")
            if w_leg % num_local_y_points != 0:
                print("FAIL: SF west leg")
                return False

            c_leg = ny_inner - jyseps2_1 - 1
            print(f"Central leg: ny_inner - jyseps2_1 - 1: {c_leg} % {num_local_y_points} = {c_leg % num_local_y_points}")
            if c_leg % num_local_y_points != 0:
                print("FAIL: SF central leg")
                return False

            s_leg1 = ny - ny_inner + 1
            print(f"South leg A: ny - ny_inner + 1: {s_leg1} % {num_local_y_points} = {s_leg1 % num_local_y_points}")
            if s_leg1 % num_local_y_points != 0:
                print("FAIL: SF south leg A")
                return False

            s_leg2 = ny - jyseps2_2 - 1
            print(f"South leg B: s_leg2 = ny - jyseps2_2 - 1: {s_leg2} % {num_local_y_points} = {s_leg2 % num_local_y_points}")
            if s_leg2 % num_local_y_points != 0:
                print("FAIL: SF south leg B")
                return False

    else:
        core = jyseps2_2 - jyseps1_1
        print(f"\nSingle Null core: {core} % {num_local_y_points} = {core % num_local_y_points}")
        if core % num_local_y_points != 0:
            print("FAIL: SN core")
            return False

    final_leg = ny - jyseps2_2 - 1
    print(f"\nFinal leg: {final_leg} % {num_local_y_points} = {final_leg % num_local_y_points}")
    if final_leg % num_local_y_points != 0:
        print("FAIL: final leg")
        return False

    print("\nPASS: decomposition is valid")
    return True


In [23]:
params = {
    "num_y_processors": 9,
    "ny": 54,
    "num_y_guards": 1,
    "jyseps1_1": 2,
    "jyseps2_1": 8,
    "jyseps1_2": 14,
    "jyseps2_2": 26,
    "ny_inner": 21,
    "topology": "SF",
}

check_bout_mesh_y_decomposition(params)



=== INPUT ===
num_y_processors = 9
ny              = 54
num_y_guards    = 1
jyseps1_1       = 2
jyseps2_1       = 8
jyseps1_2       = 14
jyseps2_2       = 26
ny_inner        = 21
topology        = SF

MYSUB = ny / NYPE = 6

--- Snowflake checks ---
Core: jyseps2_1 - jyseps1_1: 6 % 6 = 0
E leg: jyseps2_2 - ny_inner + 1: 6 % 6 = 0
N leg: ny_inner - jyseps1_2 - 1: 6 % 6 = 0
W leg: jyseps1_2 - jyseps2_1 - 1: 5 % 6 = 5
FAIL: SF west leg


False

In [25]:
def find_valid_decomposition(base_params, verbose=False):
    """
    Try to find a valid Y-decomposition by rotating all index parameters
    except ny. Uses brute-force search with ordering constraints.

    If a valid configuration is found, prints and returns it.
    Otherwise prints that no configuration is possible.
    """

    ny = base_params["ny"]
    topology = base_params["topology"]

    tested = 0

    for jyseps1_1 in range(0, ny - 4):
        for jyseps2_1 in range(jyseps1_1 + 1, ny - 3):
            for jyseps1_2 in range(jyseps2_1 + 1, ny - 2):
                for jyseps2_2 in range(jyseps1_2 + 1, ny - 1):
                    for ny_inner in range(jyseps2_1 + 1, jyseps2_2):

                        # Enforce Snowflake-specific constraint
                        if topology == "SF":
                            if not (jyseps1_1 < jyseps2_1 < jyseps1_2 < jyseps2_2):
                                continue

                        params = dict(base_params)
                        params.update({
                            "jyseps1_1": jyseps1_1,
                            "jyseps2_1": jyseps2_1,
                            "jyseps1_2": jyseps1_2,
                            "jyseps2_2": jyseps2_2,
                            "ny_inner": ny_inner,
                        })

                        tested += 1
                        ok = check_bout_mesh_y_decomposition(params)

                        if ok:
                            print("\n✅ FOUND VALID DECOMPOSITION")
                            for k, v in params.items():
                                print(f"{k:15s} = {v}")
                            print(f"\nTested {tested} combinations")
                            return params

                        if verbose and tested % 10000 == 0:
                            print(f"Tested {tested} combinations...")

    print(" NO VALID DECOMPOSITION POSSIBLE")
    print(f"ny = {ny}, topology = {topology}")
    print(f"Tested {tested} combinations")
    return None


In [29]:
def check_bout_mesh_y_decomposition(p):
    """
    Diagnostic version of checkBoutMeshYDecomposition.

    Prints every region length and modulo check so you can see
    exactly why a given configuration passes or fails.
    """

    num_y_processors = p["num_y_processors"]
    ny = p["ny"]
    num_y_guards = p["num_y_guards"]
    jyseps1_1 = p["jyseps1_1"]
    jyseps2_1 = p["jyseps2_1"]
    jyseps1_2 = p["jyseps1_2"]
    jyseps2_2 = p["jyseps2_2"]
    ny_inner = p["ny_inner"]
    topology = p["topology"]

    #print("\n=== INPUT ===")
    #for k, v in p.items():
        #print(f"{k:15s} = {v}")

    num_local_y_points = ny // num_y_processors
    #print(f"\nMYSUB = ny / NYPE = {num_local_y_points}")

    # --- Guard check ---
    if num_local_y_points < num_y_guards and num_y_processors != 1:
        #print("FAIL: ny/NYPE < MYG")
        return False

    # --- Lower leg check ---
    val = jyseps1_1
    #print(f"\nLower leg: (jyseps1_1 + 1) = {val}")
    #print(f"  {val} % {num_local_y_points} = {val % num_local_y_points}")
    if val % num_local_y_points != 0:
        #print("FAIL: lower leg")
        return False

    # --- Topology-specific ---
    if jyseps2_1 != jyseps1_2:

        if topology in ("UDN", "CDN"):
            core1 = jyseps2_1 - jyseps1_1
            #print(f"\nCore DN1: {core1} % {num_local_y_points} = {core1 % num_local_y_points}")
            if core1 % num_local_y_points != 0:
                #print("FAIL: DN core 1")
                return False

            core2 = jyseps2_2 - jyseps1_2
            #print(f"Core DN2: {core2} % {num_local_y_points} = {core2 % num_local_y_points}")
            if core2 % num_local_y_points != 0:
                #print("FAIL: DN core 2")
                return False

            leg1 = ny_inner - jyseps2_1 - 1
            #print(f"Upper leg 1: {leg1} % {num_local_y_points} = {leg1 % num_local_y_points}")
            if leg1 % num_local_y_points != 0:
                #print("FAIL: upper leg 1")
                return False

            leg2 = jyseps1_2 - ny_inner + 1
            #print(f"Upper leg 2: {leg2} % {num_local_y_points} = {leg2 % num_local_y_points}")
            if leg2 % num_local_y_points != 0:
                #print("FAIL: upper leg 2")
                return False

        elif topology == "SF":
            #print("\n--- Snowflake checks ---")

            core = jyseps2_1 - jyseps1_1
            #print(f"Core: {core} % {num_local_y_points} = {core % num_local_y_points}")
            if core % num_local_y_points != 0:
                #print("FAIL: SF core")
                return False

            ea_leg = ny_inner - 1 - jyseps1_2
            #print(f"E leg: {e_leg} % {num_local_y_points} = {e_leg % num_local_y_points}")
            if ea_leg % num_local_y_points != 0:
                #print("FAIL: SF east leg")
                return False

            #n_leg = ny_inner - jyseps1_2 - 1
            #print(f"N leg: {n_leg} % {num_local_y_points} = {n_leg % num_local_y_points}")
            #if n_leg % num_local_y_points != 0:
                #print("FAIL: SF north leg")
            #    return False

            eb_leg = jyseps2_2 - (ny_inner + 1)
            if eb_leg % num_local_y_points != 0:
                #print("FAIL: SF east leg")
                return False

            wb_leg = jyseps1_2 - jyseps2_1
            #print(f"W leg: {w_leg} % {num_local_y_points} = {w_leg % num_local_y_points}")
            if wb_leg % num_local_y_points != 0:
                #print("FAIL: SF west leg")
                return False
            
            wc_leg = ny - jyseps2_2
            #print(f"W leg: {w_leg} % {num_local_y_points} = {w_leg % num_local_y_points}")
            if wc_leg % num_local_y_points != 0:
                #print("FAIL: SF west leg")
                return False

            nce_leg = ny_inner - 1 - jyseps2_1
            #print(f"Central leg: {c_leg} % {num_local_y_points} = {c_leg % num_local_y_points}")
            if nce_leg % num_local_y_points != 0:
                #print("FAIL: SF central leg")
                return False

            s_leg = ny - ny_inner
            #print(f"South leg B: {s_leg2} % {num_local_y_points} = {s_leg2 % num_local_y_points}")
            if s_leg % num_local_y_points != 0:
                #print("FAIL: SF south leg B")
                return False

    else:
        core = jyseps2_2 - jyseps1_1
        #print(f"\nSingle Null core: {core} % {num_local_y_points} = {core % num_local_y_points}")
        if core % num_local_y_points != 0:
            #print("FAIL: SN core")
            return False

    print("\nPASS: decomposition is valid")
    return True


In [30]:
def find_valid_decomposition(base_params, verbose=False):
    """
    Try to find a valid Y-decomposition by rotating all index parameters
    except ny. Uses brute-force search with ordering constraints.

    If a valid configuration is found, prints and returns it.
    Otherwise prints that no configuration is possible.
    """

    ny = base_params["ny"]
    topology = base_params["topology"]

    tested = 0

    for jyseps1_1 in range(0, ny - 4):
        for jyseps2_1 in range(jyseps1_1 + 1, ny - 3):
            for jyseps1_2 in range(jyseps2_1 + 1, ny - 2):
                for jyseps2_2 in range(jyseps1_2 + 1, ny - 1):
                    for ny_inner in range(jyseps2_1 + 1, jyseps2_2):

                        # Enforce Snowflake-specific constraint
                        if topology == "SF":
                            if not (jyseps1_1 < jyseps2_1 < jyseps1_2 < ny_inner< jyseps2_2):
                                continue

                        params = dict(base_params)
                        params.update({
                            "jyseps1_1": jyseps1_1,
                            "jyseps2_1": jyseps2_1,
                            "jyseps1_2": jyseps1_2,
                            "jyseps2_2": jyseps2_2,
                            "ny_inner": ny_inner,
                        })

                        tested += 1
                        ok = check_bout_mesh_y_decomposition(params)

                        if ok:
                            print("\n FOUND VALID DECOMPOSITION")
                            for k, v in params.items():
                                print(f"{k:15s} = {v}")
                            print(f"\nTested {tested} combinations")
                            return params

                        if verbose and tested % 10000 == 0:
                            print(f"Tested {tested} combinations...")

    print("\n NO VALID DECOMPOSITION POSSIBLE")
    print(f"ny = {ny}, topology = {topology}")
    print(f"Tested {tested} combinations")
    return None


In [31]:
params = {
    "num_y_processors": 9,
    "ny": 54,
    "num_y_guards": 1,
    "jyseps1_1": 1,
    "jyseps2_1": 1,
    "jyseps1_2": 1,
    "jyseps2_2": 1,
    "ny_inner": 1,
    "topology": "SF",
}

find_valid_decomposition(params)



 NO VALID DECOMPOSITION POSSIBLE
ny = 54, topology = SF
Tested 2869685 combinations


In [4]:
def check_y_decomposition(ny, num_y_processors, num_y_guards,
                          jyseps1_1, jyseps2_1, jyseps1_2, jyseps2_2,
                          ny_inner):
    """
    Check if a given Y-decomposition is valid according to the updated logic.
    Returns: (success: bool, reason: str)
    """
    num_local_y_points = ny // num_y_processors

    #check that it is not trivial case
    if (ny_inner - 1 - jyseps1_2) == 0:
        return False, "ny_inner - 1 - jyseps1_2 cannot be zero."

    # Check Core Region
    if (jyseps2_1 - jyseps1_1) % num_local_y_points != 0:
        return False, f"Core region jyseps2_1-jyseps1_1 ({jyseps2_1}-{jyseps1_1} = {jyseps2_1-jyseps1_1}) must be a multiple of MYSUB ({num_local_y_points})"

    # Check E leg region
    if (jyseps2_2 - ny_inner) % num_local_y_points != 0:
        return False, f"leg region jyseps2_2-ny_inner ({jyseps2_2}-{ny_inner} = {jyseps2_2-ny_inner}) must be a multiple of MYSUB ({num_local_y_points})"

    if (ny_inner - 1 - jyseps1_2) % num_local_y_points != 0:
        return False, f"leg region ny_inner ({ny_inner}-{jyseps1_2}-1 = {ny_inner-1-jyseps1_2}) must be a multiple of MYSUB ({num_local_y_points})"

    # Check W leg region
    if (jyseps1_2 - jyseps2_1) % num_local_y_points != 0:
        return False, f"leg region jyseps1_2-jyseps2_1-1 ({jyseps1_2}-{jyseps2_1}-1 = {jyseps1_2-jyseps2_1-1}) must be a multiple of MYSUB ({num_local_y_points})"

    if (ny - jyseps2_2) % num_local_y_points != 0:
        return False, f"leg region ny-jyseps2_2-1 ({ny}-{jyseps2_2} = {ny-jyseps2_2}) must be a multiple of MYSUB ({num_local_y_points})"

    # Check central region
    if (ny_inner - 1 - jyseps2_1) % num_local_y_points != 0:
        return False, f"leg region ny_inner-jyseps2_1-1 ({ny_inner}-{jyseps2_1}-1 = {ny_inner-1-jyseps2_1}) must be a multiple of MYSUB ({num_local_y_points})"

    # Check South leg region
    if (ny - ny_inner) % num_local_y_points != 0:
        return False, f"leg region ny - ny_inner + 1 ({ny}-{ny_inner} = {ny-ny_inner}) must be a multiple of MYSUB ({num_local_y_points})"

    return True, ""


In [27]:
def find_valid_y_decomposition(ny, num_y_processors, num_y_guards,
                               jyseps1_1_range, jyseps2_1_range,
                               jyseps1_2_range, jyseps2_2_range,
                               ny_inner_range):
    """
    Try all combinations of indices within the given ranges.
    Returns a tuple of valid indices or None if impossible.
    """
    num_local_y_points = ny // num_y_processors

    for jyseps1_1 in jyseps1_1_range:
        for jyseps2_1 in jyseps2_1_range:
            for jyseps1_2 in jyseps1_2_range:
                for jyseps2_2 in jyseps2_2_range:
                    for ny_inner in ny_inner_range:
                        if (jyseps1_1 < jyseps2_1 < jyseps1_2 < ny_inner< jyseps2_2):
                            p = [ny, num_y_processors, num_y_guards,
                               jyseps1_1_range, jyseps2_1_range,
                               jyseps1_2_range, jyseps2_2_range,
                               ny_inner_range]
                            success, reason = check_y_decomposition(
                               p
                            )
                            if success:
                                return (jyseps1_1, jyseps2_1, jyseps1_2, jyseps2_2, ny_inner)
    return None  # No combination works


In [30]:
ny =9
num_y_processors = 9
num_y_guards = 1

jyseps1_1_range = range(1, ny)
jyseps2_1_range = range(1, ny)
jyseps1_2_range = range(1, ny)
jyseps2_2_range = range(1, ny)
ny_inner_range = range(1, ny)

valid = find_valid_y_decomposition(
    ny, num_y_processors, num_y_guards,
    jyseps1_1_range, jyseps2_1_range,
    jyseps1_2_range, jyseps2_2_range,
    ny_inner_range
)

print("Valid decomposition:", valid)


TypeError: unsupported operand type(s) for -: 'range' and 'int'

In [ ]:
DecompositionTestParameters{total_processors = 9, num_y_processors = 9, ny = 54, num_y_guards = 2, jyseps1_1 = 5, jyseps2_1 = 11, jyseps1_2 = 17, jyseps2_2 = 48, ny_inner = 24, topology = SF, expected_message =  }

In [29]:
def check_y_decomposition(p):
    """
    Check if a given Y-decomposition is valid according to the updated logic.
    Returns: (success: bool, reason: str)
    """

    num_y_processors = p[0]
    ny = p[1]
    num_y_guards = p[2]
    jyseps1_1 = p[3]
    jyseps2_1 = p[4]
    jyseps1_2 = p[5]
    jyseps2_2 = p[6]
    ny_inner = p[7]

    num_local_y_points = ny // num_y_processors

    #check that it is not trivial case
    if (ny_inner - 1 - jyseps1_2) == 0:
        return False, "ny_inner - 1 - jyseps1_2 cannot be zero."

    # Check Core Region
    if (jyseps2_1 - jyseps1_1) % num_local_y_points != 0:
        return False, f"Core region jyseps2_1-jyseps1_1 ({jyseps2_1}-{jyseps1_1} = {jyseps2_1-jyseps1_1}) must be a multiple of MYSUB ({num_local_y_points})"

    # Check E leg region
    if (jyseps2_2 - ny_inner + 1) % num_local_y_points != 0:
        return False, f"leg region jyseps2_2-ny_inner ({jyseps2_2}-{ny_inner} = {jyseps2_2-ny_inner}) must be a multiple of MYSUB ({num_local_y_points})"

    if (ny_inner - 1 - jyseps1_2) % num_local_y_points != 0:
        return False, f"leg region ny_inner ({ny_inner}-{jyseps1_2}-1 = {ny_inner-1-jyseps1_2}) must be a multiple of MYSUB ({num_local_y_points})"

    # Check W leg region
    if (jyseps1_2 - jyseps2_1) % num_local_y_points != 0:
        return False, f"leg region jyseps1_2-jyseps2_1-1 ({jyseps1_2}-{jyseps2_1}-1 = {jyseps1_2-jyseps2_1-1}) must be a multiple of MYSUB ({num_local_y_points})"

    if (ny - jyseps2_2 - 1) % num_local_y_points != 0:
        return False, f"leg region ny-jyseps2_2-1 ({ny}-{jyseps2_2} = {ny-jyseps2_2}) must be a multiple of MYSUB ({num_local_y_points})"

    # Check central region
    if (ny_inner - 1 - jyseps2_1) % num_local_y_points != 0:
        return False, f"leg region ny_inner-jyseps2_1-1 ({ny_inner}-{jyseps2_1}-1 = {ny_inner-1-jyseps2_1}) must be a multiple of MYSUB ({num_local_y_points})"

    # Check South leg region
    if (ny - ny_inner) % num_local_y_points != 0:
        return False, f"leg region ny - ny_inner + 1 ({ny}-{ny_inner} = {ny-ny_inner}) must be a multiple of MYSUB ({num_local_y_points})"

    return True, ""


In [21]:
params = {
    "num_y_processors": 9,
    "ny": 54,
    "num_y_guards": 1,
    "jyseps1_1": 5,
    "jyseps2_1": 11,
    "jyseps1_2": 17,
    "jyseps2_2": 48,
    "ny_inner": 24,

}

check_y_decomposition(params)

(False,
 'leg region jyseps2_2-ny_inner (48-24 = 24) must be a multiple of MYSUB (6)')

In [ ]:
def check_y_decomposition(p):
    """
    Check if a given Y-decomposition is valid according to the updated logic.
    Returns: (success: bool, reason: str)
    """

    num_y_processors = p["num_y_processors"]
    ny = p["ny"]
    num_y_guards = p["num_y_guards"]
    jyseps1_1 = p["jyseps1_1"]
    jyseps2_1 = p["jyseps2_1"]
    jyseps1_2 = p["jyseps1_2"]
    jyseps2_2 = p["jyseps2_2"]
    ny_inner = p["ny_inner"]

    num_local_y_points = ny // num_y_processors

    #check that it is not trivial case
    if (ny_inner - 1 - jyseps1_2) == 0:
        return False, "ny_inner - 1 - jyseps1_2 cannot be zero."

    # Check Core Region
    if (jyseps2_1 - jyseps1_1) % num_local_y_points != 0:
        return False, f"Core region jyseps2_1-jyseps1_1 ({jyseps2_1}-{jyseps1_1} = {jyseps2_1-jyseps1_1}) must be a multiple of MYSUB ({num_local_y_points})"

    # Check E leg region
    if (jyseps2_2 - ny_inner + 1) % num_local_y_points != 0:
        return False, f"leg region jyseps2_2-ny_inner ({jyseps2_2}-{ny_inner} = {jyseps2_2-ny_inner}) must be a multiple of MYSUB ({num_local_y_points})"

    if (ny_inner - 1 - jyseps1_2) % num_local_y_points != 0:
        return False, f"leg region ny_inner ({ny_inner}-{jyseps1_2}-1 = {ny_inner-1-jyseps1_2}) must be a multiple of MYSUB ({num_local_y_points})"

    # Check W leg region
    if (jyseps1_2 - jyseps2_1) % num_local_y_points != 0:
        return False, f"leg region jyseps1_2-jyseps2_1-1 ({jyseps1_2}-{jyseps2_1}-1 = {jyseps1_2-jyseps2_1-1}) must be a multiple of MYSUB ({num_local_y_points})"

    if (ny - jyseps2_2 - 1) % num_local_y_points != 0:
        return False, f"leg region ny-jyseps2_2-1 ({ny}-{jyseps2_2} = {ny-jyseps2_2}) must be a multiple of MYSUB ({num_local_y_points})"

    # Check central region
    if (ny_inner - 1 - jyseps2_1) % num_local_y_points != 0:
        return False, f"leg region ny_inner-jyseps2_1-1 ({ny_inner}-{jyseps2_1}-1 = {ny_inner-1-jyseps2_1}) must be a multiple of MYSUB ({num_local_y_points})"

    # Check South leg region
    if (ny - ny_inner) % num_local_y_points != 0:
        return False, f"leg region ny - ny_inner + 1 ({ny}-{ny_inner} = {ny-ny_inner}) must be a multiple of MYSUB ({num_local_y_points})"

    return True, ""



def find_valid_y_decomposition(ny, num_y_processors, num_y_guards,
                               jyseps1_1_range, jyseps2_1_range,
                               jyseps1_2_range, jyseps2_2_range,
                               ny_inner_range):
    """
    Try all combinations of indices within the given ranges.
    Returns a tuple of valid indices or None if impossible.
    """
    num_local_y_points = ny // num_y_processors

    for jyseps1_1 in jyseps1_1_range:
        for jyseps2_1 in jyseps2_1_range:
            for jyseps1_2 in jyseps1_2_range:
                for jyseps2_2 in jyseps2_2_range:
                    for ny_inner in ny_inner_range:
                        if (jyseps1_1 < jyseps2_1 < jyseps1_2 < ny_inner< jyseps2_2):
                            success, reason = check_y_decomposition(
                                ny, num_y_processors, num_y_guards,
                                jyseps1_1, jyseps2_1, jyseps1_2, jyseps2_2, ny_inner
                            )
                            if success:
                                return (jyseps1_1, jyseps2_1, jyseps1_2, jyseps2_2, ny_inner)
    return None  # No combination works

In [1]:
def check_y_decomposition(params):
    """
    Check if a given Y-decomposition is valid according to the updated logic.
    params: dict with keys ny, num_y_processors, num_y_guards,
            jyseps1_1, jyseps2_1, jyseps1_2, jyseps2_2, ny_inner
    Returns: (success: bool, reason: str)
    """
    ny = params["ny"]
    num_y_processors = params["num_y_processors"]
    num_y_guards = params["num_y_guards"]
    jyseps1_1 = params["jyseps1_1"]
    jyseps2_1 = params["jyseps2_1"]
    jyseps1_2 = params["jyseps1_2"]
    jyseps2_2 = params["jyseps2_2"]
    ny_inner = params["ny_inner"]

    num_local_y_points = ny / num_y_processors

    # Trivial case check
    if (ny_inner - 1 - jyseps1_2) == 0:
        return False, "ny_inner - 1 - jyseps1_2 cannot be zero."

    # Core region
    if (jyseps2_1 - jyseps1_1) % num_local_y_points != 0:
        return False, f"Core region ({jyseps2_1}-{jyseps1_1}) not multiple of {num_local_y_points}"

    # E leg region
    if (jyseps2_2 - ny_inner + 1) % num_local_y_points != 0:
        return False, f"E leg ({jyseps2_2}-{ny_inner+1}) not multiple of {num_local_y_points}"
    if (ny_inner - 1 - jyseps1_2) % num_local_y_points != 0:
        return False, f"E leg ({ny_inner-1}-{jyseps1_2}) not multiple of {num_local_y_points}"

    # W leg region
    if (jyseps1_2 - jyseps2_1) % num_local_y_points != 0:
        return False, f"W leg ({jyseps1_2}-{jyseps2_1}-1) not multiple of {num_local_y_points}"
    if (ny - jyseps2_2 - 1) % num_local_y_points != 0:
        return False, f"W leg ({ny}-{jyseps2_2}-1) not multiple of {num_local_y_points}"

    # Central region
    if (ny_inner - 1 - jyseps2_1) % num_local_y_points != 0:
        return False, f"Central region ({ny_inner-1}-{jyseps2_1}) not multiple of {num_local_y_points}"

    # South leg region
    if (ny -1 - ny_inner+1) % num_local_y_points != 0:
        return False, f"South leg ({ny}-{ny_inner}) not multiple of {num_local_y_points}"

    return True, ""


def find_valid_y_decomposition(ny, num_y_processors, num_y_guards,
                               jyseps1_1_range, jyseps2_1_range,
                               jyseps1_2_range, jyseps2_2_range,
                               ny_inner_range):
    """
    Iterate through all combinations of indices and return the first valid one.
    """
    for jyseps1_1 in jyseps1_1_range:
        for jyseps2_1 in jyseps2_1_range:
            for jyseps1_2 in jyseps1_2_range:
                for jyseps2_2 in jyseps2_2_range:
                    for ny_inner in ny_inner_range:
                        # Ensure indices are in increasing order
                        if not (jyseps1_1 < jyseps2_1 < jyseps1_2 < ny_inner < jyseps2_2):
                            continue

                        params = {
                            "ny": ny,
                            "num_y_processors": num_y_processors,
                            "num_y_guards": num_y_guards,
                            "jyseps1_1": jyseps1_1,
                            "jyseps2_1": jyseps2_1,
                            "jyseps1_2": jyseps1_2,
                            "jyseps2_2": jyseps2_2,
                            "ny_inner": ny_inner
                        }

                        success, reason = check_y_decomposition(params)
                        if success:
                            return jyseps1_1, jyseps2_1, jyseps1_2, jyseps2_2, ny_inner
    print("No valid decomposition found.")
    return None  # No combination found


In [9]:
ny = 10

num_y_processors = 5
num_y_guards = 1

jyseps1_1_range = range(1, ny)
jyseps2_1_range = range(1, ny)
jyseps1_2_range = range(1, ny)
jyseps2_2_range = range(1, ny)
ny_inner_range = range(1, ny)

valid = find_valid_y_decomposition(
    ny, num_y_processors, num_y_guards,
    jyseps1_1_range, jyseps2_1_range,
    jyseps1_2_range, jyseps2_2_range,
    ny_inner_range
)

print("Valid decomposition:", valid)

Valid decomposition: (1, 3, 5, 9, 8)
